# NABirds Class Forest Visualization (PyVis)

This notebook reads `classes.txt` and `hierarchy.txt`, builds the class hierarchy, and renders an interactive graph with PyVis.

Tip: start with a subtree (e.g., `ROOT_TO_VIEW = 22`) for faster rendering, then switch to `ROOT_TO_VIEW = 0` for the full hierarchy.

In [3]:
from pathlib import Path
from collections import deque

import pandas as pd
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display

DATA_DIR = Path('NABirds Dataset/nabirds')
CLASSES_PATH = DATA_DIR / 'classes.txt'
HIERARCHY_PATH = DATA_DIR / 'hierarchy.txt'

assert CLASSES_PATH.exists(), f'Missing file: {CLASSES_PATH}'
assert HIERARCHY_PATH.exists(), f'Missing file: {HIERARCHY_PATH}'

In [4]:
# Load data
class_rows = []
with open(CLASSES_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid, cname = line.split(' ', 1)
        class_rows.append((int(cid), cname))

classes_df = pd.DataFrame(class_rows, columns=['class_id', 'class_name'])
hierarchy_df = pd.read_csv(HIERARCHY_PATH, sep=r'\s+', header=None, names=['child_class_id', 'parent_class_id'])

# Build parent -> child directed graph
G = nx.DiGraph()

class_name = dict(zip(classes_df['class_id'], classes_df['class_name']))
for cid, cname in class_name.items():
    G.add_node(int(cid), label=f'{cid}: {cname}')

for _, row in hierarchy_df.iterrows():
    child = int(row['child_class_id'])
    parent = int(row['parent_class_id'])
    G.add_edge(parent, child)

# Compute depth from root sentinel (0)
depth = {0: 0}
q = deque([0])
while q:
    node = q.popleft()
    for child in G.successors(node):
        if child not in depth:
            depth[child] = depth[node] + 1
            q.append(child)

nx.set_node_attributes(G, depth, name='depth')

num_roots = sum(1 for _, p in hierarchy_df.values if p == 0)
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'Top-level groups (children of 0): {num_roots}')
print(f'Max depth from 0: {max(depth.values())}')


Nodes: 1011
Edges: 1010
Top-level groups (children of 0): 22
Max depth from 0: 4


In [5]:
def subtree_nodes(graph: nx.DiGraph, root: int, max_depth=None):
    """Return nodes in subtree rooted at root (including root)."""
    result = set()
    q = deque([(root, 0)])
    while q:
        node, d = q.popleft()
        if node in result:
            continue
        result.add(node)
        if max_depth is not None and d >= max_depth:
            continue
        for child in graph.successors(node):
            q.append((child, d + 1))
    return result

def render_pyvis(graph: nx.DiGraph, out_html='nabirds_forest.html'):
    net = Network(height='900px', width='100%', directed=True, bgcolor='white', font_color='black')

    # Hierarchical layout works well for taxonomy trees.
    net.set_options('''
    var options = {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "UD",
          "sortMethod": "directed",
          "nodeSpacing": 180,
          "levelSeparation": 140,
          "treeSpacing": 280
        }
      },
      "physics": {
        "enabled": false
      },
      "edges": {
        "arrows": {
          "to": {"enabled": true}
        },
        "smooth": false,
        "color": {
          "inherit": false
        }
      },
      "interaction": {
        "hover": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }
    ''')

    for n, attrs in graph.nodes(data=True):
        d = attrs.get('depth', 0)
        out_deg = graph.out_degree(n)
        node_label = attrs.get('label', str(n))

        # Emphasize internal grouping nodes and root.
        if n == 0:
            color = '#264653'
        elif out_deg > 0:
            color = '#2A9D8F'
        else:
            color = '#E9C46A'

        net.add_node(
            int(n),
            label=node_label,
            title=f'id={n}<br>depth={d}<br>children={out_deg}',
            color=color,
            level=int(d),
            size=10 + min(out_deg, 12),
        )

    for u, v in graph.edges():
        net.add_edge(int(u), int(v), color='#8D99AE')

    net.write_html(out_html, notebook=False, open_browser=False)
    display(IFrame(src=out_html, width='100%', height='920px'))
    print(f'Wrote: {out_html}')

In [6]:
# Choose what to render:
# - ROOT_TO_VIEW = 0 renders the full hierarchy
# - ROOT_TO_VIEW = 22 renders 'Perching Birds' subtree, etc.
ROOT_TO_VIEW = 0
MAX_DEPTH = None  # e.g., set to 4 for a shallower preview

nodes = subtree_nodes(G, ROOT_TO_VIEW, max_depth=MAX_DEPTH)
H = G.subgraph(nodes).copy()

print(f'Rendering subtree rooted at {ROOT_TO_VIEW} with {H.number_of_nodes()} nodes and {H.number_of_edges()} edges')
render_pyvis(H, out_html='nabirds_forest.html')

Rendering subtree rooted at 0 with 1011 nodes and 1010 edges


Wrote: nabirds_forest.html
